# 観点ごとにポジティブ・ネガティブを判定する（アスペクトベース感情分析）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_classification/absa_ja.ipynb)

NLP クックブック 11-24 ／ テキスト分類

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

「売上は伸びたが利益は減った」のような文で、文全体ではなく観点（アスペクト）ごとに、売上=ポジ、利益=ネガと判定したい。有価証券報告書の ABSA データセット chABSA を使う。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

# chABSA (TIS 株式会社 / chakki-works、CC BY 4.0、https://github.com/chakki-works/chABSA-dataset) を第三者が Hugging Face に複製したもの。
# README の配布 URL はリンク切れのため、複製の版を固定して使う (複製が削除されると読めなくなる)
ds = load_dataset("zenless-lab/chABSA", revision="254de3bf9771bcafb40812cecd33e6cdb234bb49")
sents = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
# 1 行 = (文, 観点の語, 極性)。from / to は文中の文字位置
df = pd.DataFrame([{"company": s.document_id, "sid": i, "sentence": s.sentence, "target": o["text"],
                    "start": o["from"], "end": o["to"], "label": o["polarity"]}
                   for i, s in enumerate(sents.itertuples()) for o in s.opinions])
print(len(sents), "文 /", len(df), "観点", df["label"].value_counts().to_dict())
df = df[df["label"] != "neutral"].reset_index(drop=True)   # 中立はごく少ないので除き、ポジ・ネガの 2 値にする

# 同じ会社の文が学習と評価の両方に入らないよう、会社単位で分ける
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(df, groups=df["company"]))
train, test = df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)
test["mixed"] = test.groupby("sid")["label"].transform("nunique") > 1   # ポジとネガの観点が同居する文か
print("学習:", len(train), " 評価:", len(test), " うち対比の文の観点:", int(test["mixed"].sum()))
ex = test[test["mixed"]].iloc[0]
print(ex["sentence"]); print(test[test["sid"] == ex["sid"]][["target", "label"]].to_string(index=False))

3215 文 / 7723 観点 {'positive': 4334, 'negative': 3131, 'neutral': 258}
学習: 6016  評価: 1449  うち対比の文の観点: 522
これに加え、仙台駅等の店舗の売上が好調であったことなどにより、売上高は前期比0.3％増の417,113百万円となりましたが、工事支障による閉店の影響などにより、営業利益は前期比6.0％減の32,990百万円となりました
target    label
   仙台駅 positive
   売上高 positive
  営業利益 negative


In [3]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import pipeline

def report(name, pred):
    y, p = test["label"].to_numpy(), np.asarray(pred)
    m = test["mixed"].to_numpy()
    print(f"{name:24} 正解率 {accuracy_score(y, p):.3f}  macro-F1 {f1_score(y, p, average='macro'):.3f}"
          f"  | 対比の文での正解率 {accuracy_score(y[m], p[m]):.3f}")

# (1) 文全体の感情分析 (この章の「日本語の感情分析 (ポジネガ判定) をする」と同じモデル) の結果を、文中のすべての観点に配る
sent_clf = pipeline("text-classification", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")
uniq = test.drop_duplicates("sid")
out = sent_clf(list(uniq["sentence"]), top_k=None, truncation=True, batch_size=32)
sent_pred = {sid: max((r for r in rs if r["label"] != "neutral"), key=lambda r: r["score"])["label"]   # ポジとネガの高いほう
             for sid, rs in zip(uniq["sid"], out)}
report("文全体の感情分析", test["sid"].map(sent_pred))

# 文ごとに 1 つのラベルしか出せない方法の上限: 各文で多数派の極性を (正解を見て) 全観点に付けた場合
majority = test.groupby("sid")["label"].agg(lambda s: s.mode()[0])
report("文単位の上限 (正解の多数派)", test["sid"].map(majority))

# 観点の語と文だけを見る方法の上限: 同じ文の同じ位置の語に、カテゴリ違い (例: business#sales と business#profit) で逆の極性が付いた行がある
span = test.groupby(["sid", "start", "end"])["label"]
print("同じ語に逆の極性が付いた行:", int((span.transform("nunique") > 1).sum()))
report("語と文だけで判定する上限", span.transform(lambda s: s.mode()[0]))

文全体の感情分析                 正解率 0.649  macro-F1 0.638  | 対比の文での正解率 0.515
文単位の上限 (正解の多数派)          正解率 0.865  macro-F1 0.865  | 対比の文での正解率 0.625
同じ語に逆の極性が付いた行: 34
語と文だけで判定する上限             正解率 0.988  macro-F1 0.988  | 対比の文での正解率 0.967


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# (2) 観点の語から次の読点・句点までを取り出す。日本語では「売上高は増加し、」のように、観点の後ろの述語に評価が出やすい
def window(r):
    rest = r.sentence[r.start:]
    cut = min([i for i in (rest.find("、", len(r.target)), rest.find("。")) if i >= 0], default=len(rest))
    return rest[:cut]

def tfidf_lr():
    return make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True), LogisticRegression(C=10, max_iter=1000))

whole = tfidf_lr().fit(train["sentence"], train["label"])                 # 比較用: 同じデータで学習した文全体の分類器
local = tfidf_lr().fit(train.apply(window, axis=1), train["label"])
report("TF-IDF (文全体)", whole.predict(test["sentence"]))
report("TF-IDF (観点の後ろの区間)", local.predict(test.apply(window, axis=1)))
print(test.head(3).apply(window, axis=1).tolist())

TF-IDF (文全体)             正解率 0.815  macro-F1 0.813  | 対比の文での正解率 0.569
TF-IDF (観点の後ろの区間)        正解率 0.836  macro-F1 0.834  | 対比の文での正解率 0.822
['業務市場におきましても高速道路等でのＩＰネットワーク対応インターホンシステムへの取替え活動が功を奏し', '売上高は19,069百万円(前期比29.0％増)', '営業利益9,046百万円(同29.2％増)']


In [5]:
import torch
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

# (3) 「観点の語」と「観点に印を付けた文」を文ペアにして BERT を微調整する (T4 で 5 分程度)
name = "tohoku-nlp/bert-base-japanese-v3"   # この章の「BERT をファインチューニングして文書分類する」と同じモデル (Apache 2.0)
tok = AutoTokenizer.from_pretrained(name)

def to_dataset(d):
    marked = [r.sentence[:r.start] + "【" + r.target + "】" + r.sentence[r.end:] for r in d.itertuples()]   # 同じ語が 2 回出ても区別できる
    labels = (d["label"] == "positive").astype(int).tolist() if "label" in d else [0] * len(d)
    ds = Dataset.from_dict({"aspect": list(d["target"]), "text": marked, "label": labels})
    return ds.map(lambda b: tok(b["aspect"], b["text"], truncation="only_second", max_length=192), batched=True, remove_columns=["aspect", "text"])

set_seed(0)   # 分類ヘッドの初期化も固定する
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2, id2label={0: "negative", 1: "positive"},
                                                           label2id={"negative": 0, "positive": 1})
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
args = TrainingArguments(output_dir="absa-bert", num_train_epochs=3, learning_rate=3e-5, warmup_steps=50, weight_decay=0.01,
                         per_device_train_batch_size=32, per_device_eval_batch_size=64, bf16=BF16, fp16=not BF16,
                         save_strategy="no", logging_steps=50, report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train), data_collator=DataCollatorWithPadding(tok))
trainer.train()

def predict(d):
    return np.where(trainer.predict(to_dataset(d)).predictions.argmax(-1) == 1, "positive", "negative").tolist()

report("文ペアの微調整 (BERT)", predict(test))
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Step,Training Loss
50,0.635343
100,0.348660
150,0.197747
200,0.191312
250,0.120557
300,0.128524
350,0.095695
400,0.096706
450,0.080865
500,0.097686


文ペアの微調整 (BERT)           正解率 0.967  macro-F1 0.967  | 対比の文での正解率 0.937
GPU メモリの最大使用量: 4.7 GB


In [6]:
# 新しい文で試す。観点の語を与えると、文中の位置を探して判定する
def absa(sentence, targets):
    d = pd.DataFrame([{"sentence": sentence, "target": t, "start": sentence.index(t), "end": sentence.index(t) + len(t)} for t in targets])
    return dict(zip(targets, predict(d)))

print(absa("売上高は前期を上回ったものの、原材料価格の高騰により営業利益は減少した。", ["売上高", "原材料価格", "営業利益"]))
print(absa("国内の販売は低調だったが、海外の販売は好調に推移した。", ["国内の販売", "海外の販売"]))
print("文全体の感情分析:", sent_clf("国内の販売は低調だったが、海外の販売は好調に推移した。")[0])

{'売上高': 'positive', '原材料価格': 'negative', '営業利益': 'negative'}


{'国内の販売': 'negative', '海外の販売': 'positive'}
文全体の感情分析: {'label': 'neutral', 'score': 0.3933875858783722}


- 文全体の感情分析は 1 つの文に 1 つのラベルしか出せない。評価の 1,449 観点のうち 522 は対比の文（ポジとネガが同居）にあり、正解を見て文ごとに多数派を選んでも、対比の文の正解率は 0.625 が上限になる。
- 同じデータで学習した TF-IDF でも、文全体が入力だと対比の文は 0.569 だが、観点の後ろの区間だけにすると 0.822 に上がる。入力に「どの観点か」を含めることが重要である。
- 文ペア（観点, 観点に【】で印を付けた文）で BERT を微調整すると、全体 0.967、対比の文 0.937 になった。1 回の実行の値で、種を固定しても GPU の計算は完全には再現しない（set_seed を入れる前を含む検証の 3 回の実行では、全体で 0.967〜0.971 だった）。評価件数から見た 95% 区間はおよそ全体 ±0.009、対比の文 ±0.02 で、手法どうしの差はこれよりずっと大きい。
- 同じ文の同じ語に、カテゴリ違い（例: 「運輸業」の business#sales はポジ、business#profit はネガ）で逆の極性が付いた行が評価に 34 ある。語と文だけを入力にする方法では、正解率は全体 0.988、対比の文 0.967 が上限になる。カテゴリも区別したいなら、カテゴリを入力に加える。
- 新しい決算の文 2 つ（観点 5 つ）では期待どおりに判定した（正解ラベルのない例）。学習データは有価証券報告書だけなので、分野外の文は当てにならない。検証では「料理は美味しいが接客が悪い」の「料理」の判定が、同じコードの実行ごとにポジとネガで変わった。
- 数値は A5000 で float16 の経路（`BF16 = False`、T4 と同じ dtype）で実行したもので、GPU メモリの最大使用量は 4.7 GB だった。(1) と (2) は CPU だけで動く。この章の「ラベル付きデータなしで分類する（ゼロショット分類、NLI）」で「売上高は好調だ」という仮説の含意を判定させたり、LLM に観点ごとの極性を JSON で答えさせたりする方法もある。
- chABSA は TIS の久保・中山によるデータセットで、2016 年度の有価証券報告書 230 社分の「経営成績の概要」のうち、観点の付いた 3,215 文（全 6,119 文のうち）に観点の語、カテゴリ（company#sales など）、極性を付けたもの（CC BY 4.0、https://github.com/chakki-works/chABSA-dataset）。ここでは Hugging Face 上の第三者による複製を使った（文数・観点数は論文と一致する。複製にはライセンスの記載がないので、元データの CC BY 4.0 に従う）。
- 観点の語は与えられている前提で、極性だけを判定した。文から観点の語を見つけるのは系列ラベリングの問題になる（固有表現抽出の章の「文字位置のアノテーションを BIO タグ列に変換する」「BERT をファインチューニングして日本語の固有表現抽出器を作る」）。